In [1]:
import math
import torch
from einops import einsum

# Test Linear

In [2]:
from modules import LinearLayer

linear1 = LinearLayer(in_features=4, out_features=3)
linear1.W.shape

torch.Size([3, 4])

In [3]:
x1d = torch.randn(4)           # (d_in)
x2d = torch.randn(5, 4)        # (batch, d_in)
x3d = torch.randn(2, 5, 4)     # (batch, seq, d_in)

print(linear1(x1d).shape)
print(linear1(x2d).shape)
print(linear1(x3d).shape)

torch.Size([3])
torch.Size([5, 3])
torch.Size([2, 5, 3])


# Test Embedding

In [4]:
from modules import EmbeddingLayer

emb = EmbeddingLayer(10, 4)  # vocab=10, d_model=4

x0 = torch.tensor(3)                     # scalar
x1 = torch.tensor([1, 2, 3])              # (seq_len,)
x2 = torch.tensor([[1, 2], [3, 4]])       # (batch, seq_len)

print(emb(x0).shape, emb(x0).grad_fn)
print(emb(x1).shape, emb(x1).grad_fn)
print(emb(x2).shape, emb(x2).grad_fn)

torch.Size([4]) <SelectBackward0 object at 0x103f7b040>
torch.Size([3, 4]) <IndexBackward0 object at 0x125ce9c00>
torch.Size([2, 2, 4]) <IndexBackward0 object at 0x103f7b040>


In [5]:
emb(x2).grad_fn

# Test RMSNorm

In [6]:
from modules import RMSNormLayer

norm = RMSNormLayer(4)
x = torch.randn(2, 3, 4)  # (batch, seq_len, d_model)
out = norm(x)

print(out.shape)  # should be (2, 3, 4) — same as input
print(out.dtype)  # should match x.dtype
print(x.dtype)

torch.Size([2, 3, 4])
torch.float32
torch.float32


# Test SwiGLU

In [7]:
from modules import SwiGLU

swiglu = SwiGLU(d_model=8, d_ff=24)
x = torch.randn(2, 5, 8)
out = swiglu(x)

print(out.shape)  # what do you expect, and what do you get?

torch.Size([2, 5, 8])


In [8]:
buf = torch.randn(10, 4)  # (max_seq_len, d_k//2)
pos = torch.tensor([0, 1, 2])  # (seq_len,)
print(buf[pos].shape)  # what do you get?

pos_batched = torch.tensor([[0,1,2],[3,4,5]])  # (batch, seq_len)
print(buf[pos_batched].shape)  # what do you get?

torch.Size([3, 4])
torch.Size([2, 3, 4])


# Test RoPE

In [9]:
from modules import RoPE

# Test Softmax

In [10]:
from modules import SoftmaxLayer

x = torch.randn(size=(2, 2, 4))
SoftmaxLayer(x, dim=2)

tensor([[[0.7571, 0.1016, 0.0417, 0.0996],
         [0.0640, 0.7094, 0.1669, 0.0597]],

        [[0.0395, 0.4289, 0.0370, 0.4946],
         [0.0487, 0.4153, 0.2745, 0.2615]]])

# Test Attention

In [11]:
from modules import AttentionLayer

In [12]:
Q = torch.randn(2, 3, 2)
K = torch.randn(2, 4, 2)
V = torch.randn(4,2)
mask = torch.randint(high=2, size=(3,4))
mask = torch.where(mask == 1, True, False)
output = AttentionLayer(Q, K, V, mask)
output

tensor([[[ 0.2970,  0.4867],
         [-0.0631,  0.2325],
         [ 0.3846,  0.4580]],

        [[ 0.1900,  0.4811],
         [-0.0631,  0.2325],
         [ 0.1911,  0.4580]]])

# RoPE debug

In [ ]:
theta = 10000
d_k = 6
max_seq_len = 10
inv_freq = 1.0 / (theta ** (torch.arange(0, d_k, 2, dtype=torch.float32) / d_k))
inv_freq

tensor([1.0000, 0.0464, 0.0022])

In [16]:
t = torch.arange(max_seq_len, dtype=torch.float32)
t

tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

In [23]:
thetas = einsum(t, inv_freq, "i,j -> i j")
thetas

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03],
        [4.0000e+00, 1.8566e-01, 8.6177e-03],
        [5.0000e+00, 2.3208e-01, 1.0772e-02],
        [6.0000e+00, 2.7850e-01, 1.2927e-02],
        [7.0000e+00, 3.2491e-01, 1.5081e-02],
        [8.0000e+00, 3.7133e-01, 1.7235e-02],
        [9.0000e+00, 4.1774e-01, 1.9390e-02]])

In [24]:
cos_buffer = torch.cos(thetas)
sin_buffer = torch.sin(thetas)
print(cos_buffer), print(sin_buffer)

tensor([[ 1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9989,  1.0000],
        [-0.4161,  0.9957,  1.0000],
        [-0.9900,  0.9903,  1.0000],
        [-0.6536,  0.9828,  1.0000],
        [ 0.2837,  0.9732,  0.9999],
        [ 0.9602,  0.9615,  0.9999],
        [ 0.7539,  0.9477,  0.9999],
        [-0.1455,  0.9318,  0.9999],
        [-0.9111,  0.9140,  0.9998]])
tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0464,  0.0022],
        [ 0.9093,  0.0927,  0.0043],
        [ 0.1411,  0.1388,  0.0065],
        [-0.7568,  0.1846,  0.0086],
        [-0.9589,  0.2300,  0.0108],
        [-0.2794,  0.2749,  0.0129],
        [ 0.6570,  0.3192,  0.0151],
        [ 0.9894,  0.3629,  0.0172],
        [ 0.4121,  0.4057,  0.0194]])


(None, None)

In [ ]:
class RoPE(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None):
        super(RoPE, self).__init__()

        self.theta = theta
        self.d_k = d_k
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (theta ** (torch.arange(0, d_k, 2, dtype=torch.float32, device=device) / d_k))
        t = torch.arange(max_seq_len, dtype=torch.float32, device=device)
        thetas = einsum(t, inv_freq, "i,j -> i j")

        self.register_buffer("cos_buffer", torch.cos(thetas), persistent=False)
        self.register_buffer("sin_buffer", torch.sin(thetas), persistent=False)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor):
        cos_sliced = self.cos_buffer[token_positions].to(dtype=x.dtype)
        sin_sliced = self.sin_buffer[token_positions].to(dtype=x.dtype)

        x_pairs = x.reshape(*x.shape[:-1], self.d_k // 2, 2)
        x0 = x_pairs[...,0::2]
        x1 = x_pairs[...,1::2]

        out0 = x0 * cos_sliced - x1 * sin_sliced
        out1 = x0 * sin_sliced + x1 * cos_sliced

        out_combined = torch.stack([out0, out1], dim=-1)
        return out_combined.reshape(x.shape)